# Tutorial: Memproses Data Absensi dengan "Pairing" dan "Clustering Shift"

Notebook ini dibuat agar Anda bisa belajar langkah demi langkah bagaimana mengolah data mentah (*raw data*) dari mesin absensi menjadi rekap data kehadiran yang rapi dan siap baca.

In [ ]:
import pandas as pd
import re

### 1. Membaca Data Mentah dari File SQL
Kita mengekstrak baris `INSERT INTO` menggunakan `Regex` menjadi bentuk tabel.

In [ ]:
def parse_sql_dump(filepath):
    data = []
    pattern = re.compile(r"\('([^']*)',\s*'([^']*)',\s*'([^']*)',\s*(\d+),\s*'([^']*)'\)")
    
    with open(filepath, 'r', encoding='utf-8') as file:
        for line in file:
            if line.strip().startswith("('"):
                matches = pattern.findall(line)
                for match in matches:
                    area, pin, waktu, sts, waktu_tarik = match
                    data.append({
                        'Area': area,
                        'PIN': pin,
                        'Waktu_Scan': waktu,
                        'Status': int(sts)
                    })
    return pd.DataFrame(data)

df = parse_sql_dump('t_absensi_solutions_fp.sql')
df['Waktu_Scan'] = pd.to_datetime(df['Waktu_Scan'])
df['Tanggal'] = df['Waktu_Scan'].dt.date
df.head()

,Area,PIN,Waktu_Scan,Status,Tanggal
0,BF,0189,2026-06-19 08:00:13,0,2026-06-19
1,BF,0189,2026-06-19 19:43:51,1,2026-06-19
2,BF,0189,2026-06-20 07:59:32,0,2026-06-20
3,BF,0189,2026-06-20 16:52:08,0,2026-06-20
4,BF,0189,2026-06-22 08:01:44,1,2026-06-22


### 2. Proses Logika "Pairing" Absensi (Group By)
Kita mengelompokkan (`groupby`) data berdasarkan **Karyawan (PIN)** dan **Hari (Tanggal)** untuk mendapatkan Jam Masuk (scan pertama) dan Keluar (scan terakhir).

In [ ]:
df_rekap = df.groupby(['PIN', 'Tanggal']).agg(
    Jam_Masuk=('Waktu_Scan', 'min'),   
    Jam_Keluar=('Waktu_Scan', 'max'),  
    Total_Scan=('Waktu_Scan', 'count') 
).reset_index()

df_rekap['Jam_Masuk'] = df_rekap['Jam_Masuk'].dt.strftime('%H:%M:%S')
df_rekap['Jam_Keluar'] = df_rekap['Jam_Keluar'].dt.strftime('%H:%M:%S')

# Kosongkan Jam Keluar jika hanya 1x absen (lupa absen pulang)
df_rekap.loc[df_rekap['Total_Scan'] == 1, 'Jam_Keluar'] = "-"
df_rekap.head()

,PIN,Tanggal,Jam_Masuk,Jam_Keluar,Total_Scan
0,0189,2026-06-19,08:00:13,19:43:51,2
1,0189,2026-06-20,07:59:32,16:52:08,2
2,0189,2026-06-22,08:01:44,18:05:59,2
3,0189,2026-06-23,07:59:37,18:55:29,2
4,0189,2026-06-24,07:58:30,19:28:24,2


### 3. Clustering Shift berdasarkan Toleransi Jam Masuk
Karena karyawan Shift 1 (08:00 - 16:00) pasti akan absen **sebelum** jam 08:00 (misalnya jam 07:45 atau bahkan 06:30), kita harus memberikan **rentang toleransi (Threshold)** agar mereka tidak salah masuk ke Shift 3.

- **Shift 1 (08:00-16:00):** Toleransi absen masuk pukul `04:00` s/d `11:59`
- **Shift 2 (16:00-00:00):** Toleransi absen masuk pukul `12:00` s/d `19:59`
- **Shift 3 (00:00-08:00):** Toleransi absen masuk pukul `20:00` s/d `03:59`

In [ ]:
def tentukan_shift(jam_masuk):
    if pd.isna(jam_masuk):
        return "Tidak Diketahui"
        
    # Ambil angka jamnya saja (2 karakter pertama)
    jam = int(str(jam_masuk)[:2])
    
    if 4 <= jam < 12:
        return "Shift 1 (08:00-16:00)"
    elif 12 <= jam < 20:
        return "Shift 2 (16:00-00:00)"
    else: # 20:00 malam s/d 03:59 pagi
        return "Shift 3 (00:00-08:00)"

df_rekap['Shift'] = df_rekap['Jam_Masuk'].apply(tentukan_shift)
df_rekap.head()

,PIN,Tanggal,Jam_Masuk,Jam_Keluar,Total_Scan,Shift
0,0189,2026-06-19,08:00:13,19:43:51,2,Shift 1 (08:00-16:00)
1,0189,2026-06-20,07:59:32,16:52:08,2,Shift 1 (08:00-16:00)
2,0189,2026-06-22,08:01:44,18:05:59,2,Shift 1 (08:00-16:00)
3,0189,2026-06-23,07:59:37,18:55:29,2,Shift 1 (08:00-16:00)
4,0189,2026-06-24,07:58:30,19:28:24,2,Shift 1 (08:00-16:00)


### 4. Pairing Nama Karyawan dan Ekspor Hasil
Langkah terakhir adalah memetakan PIN ke Nama Asli lalu mengekspornya.

In [ ]:
database_karyawan = {
    '0189': 'Budi Santoso',
    '0190': 'Andi Kurniawan',
}
df_rekap['Nama_Karyawan'] = df_rekap['PIN'].map(database_karyawan).fillna("Karyawan " + df_rekap['PIN'])

# Susun kolom
df_final = df_rekap[['PIN', 'Nama_Karyawan', 'Tanggal', 'Shift', 'Jam_Masuk', 'Jam_Keluar', 'Total_Scan']]

# Distribusi Shift Baru
print(df_final['Shift'].value_counts())

# Export
df_final.to_csv('Belajar_Pairing_Absensi_Dengan_Shift_Revisi.csv', index=False)
df_final.head(15)

Shift
Shift 1 (08:00-16:00)    24053
Shift 3 (00:00-08:00)     7528
Shift 2 (16:00-00:00)     2936
Name: count, dtype: int64


,PIN,Nama_Karyawan,Tanggal,Shift,Jam_Masuk,Jam_Keluar,Total_Scan
0,0189,Budi Santoso,2026-06-19,Shift 1 (08:00-16:00),08:00:13,19:43:51,2
1,0189,Budi Santoso,2026-06-20,Shift 1 (08:00-16:00),07:59:32,16:52:08,2
2,0189,Budi Santoso,2026-06-22,Shift 1 (08:00-16:00),08:01:44,18:05:59,2
3,0189,Budi Santoso,2026-06-23,Shift 1 (08:00-16:00),07:59:37,18:55:29,2
4,0189,Budi Santoso,2026-06-24,Shift 1 (08:00-16:00),07:58:30,19:28:24,2
5,0189,Budi Santoso,2026-06-25,Shift 1 (08:00-16:00),07:59:38,19:02:35,2
6,0189,Budi Santoso,2026-06-26,Shift 1 (08:00-16:00),07:59:26,18:06:18,2
7,0189,Budi Santoso,2026-06-29,Shift 1 (08:00-16:00),07:59:10,18:53:49,2
8,0189,Budi Santoso,2026-06-30,Shift 1 (08:00-16:00),07:59:48,19:16:00,2
9,0189,Budi Santoso,2026-07-01,Shift 1 (08:00-16:00),08:00:15,18:57:36,2
